In [0]:
%pip install langchain-community
%pip install databricks-langchain
%pip install databricks-agents
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.

*** WARNING: max output size exceeded, skipping output. ***

ment already satisfied: requests-toolbelt>=1.0.0 in /local_disk0/.ephemeral_nfs/envs/pythonEnv-a9d1f140-b633-4343-b87c-a0d838ca37be/lib/python3.11/site-packages (from langsmith>=0.1.17->langchain>=0.3.0->databricks-langchain) (1.0.0)
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
model = "databricks-meta-llama-3-3-70b-instruct"
vector_search_endpoint = "test"
catalog = "rag_demo"
db = "default"
vector_search_index = "fl_studio_idx"

chain_config = {
    "llm_model_serving_endpoint_name": model,  # the foundation model we want to use
    "vector_search_endpoint_name": vector_search_endpoint,  # the endoint we want to use for vector search
    "vector_search_index": f"{catalog}.{db}.{vector_search_index}",
    "llm_prompt_template": """You are a expert in the mustic software FL Studio meant to help my retiried father Jeff learn the software. Some pieces of context may be irrelevant, in which case you should not use them to form the answer. If you do not know the answer to a question just say I do not know. \n\nContext: {context}""",
}

In [0]:
from databricks.vector_search.client import VectorSearchClient
from databricks_langchain.vectorstores import DatabricksVectorSearch
from langchain.schema.runnable import RunnableLambda
from langchain_core.output_parsers import StrOutputParser
import mlflow

## Enable MLflow Tracing
mlflow.langchain.autolog()

## Load the chain's configuration
model_config = mlflow.models.ModelConfig(development_config=chain_config)

## Turn the Vector Search index into a LangChain retriever
vector_search_as_retriever = DatabricksVectorSearch(
    endpoint=model_config.get("vector_search_endpoint_name"),
    index_name=model_config.get("vector_search_index"),
    columns=["id", "chunk_text"],
).as_retriever(search_kwargs={"k": 3})

# Method to format the docs returned by the retriever into the prompt (keep only the text from chunks)
def format_context(docs):
    chunk_contents = [f"Passage: {d.page_content}\n" for d in docs]
    return "".join(chunk_contents)

#Let's try our retriever chain:
relevant_docs = (vector_search_as_retriever | RunnableLambda(format_context)| StrOutputParser()).invoke('What is FL Studio?')

displayHTML(relevant_docs)

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.


Passage: 
 
 
 
 
 
 
The term ' Soundcard ' can be confusing. It comes from the days when most 
computers had a separate soundcard. These days most use a chip on the 
motherboard, or it may be an external device connec ted by USB. A better 
term is ‘Audio Interface’ - any device that makes t he sound you hear from 
your speakers. The Audio Driver is the software interface between the 
operating system and the device. The driver tells F L Studio what 
inputs/outputs and what sample-rates the Audio Interface can support. 
 
 10 
4. The FL Studio desktop has a number of windows, most of these are movable (with overlap), 
resizable, zoomable and switchable, so if a window isn't visible open it with the Shortcut toolbar or 
the function key noted in brackets. The main window s involved in FL Studio music creation are 
- Channel Rack (F6 ), Piano roll (F7 ). Mixer (F9 ) and Playlist (F5 ). The Browser (Alt+F8 ) is 
used to access audio files, plugins & presets (see the Options > File settings to add folders 
elsewhere on your computer to the Browser). NOTE: If you ever need to reset the position of all 
windows to their default use ( Ctrl+Shift+H ) or see the View menu options. 
 
 
Video Tutorials! A picture is worth 1000 words, so a video must be 
worth, like, BILLIONS…once you are registered pop o ver to the FL 
website and watch hours of cool video tutorials, th is is really useful, so 
I’ll SHOUT – 
http://www.youtube.com/user/imageline 
 
 
 11 
MAKE SOME NOISE 
 
 
Press the Play Button (make sure the switch to the left is in 
SONG mode) to Play a project . It’s definitely worth checking the
Passage: or functions. The manual will open showing 
the last thing you clicked in FL Studio (it’s context sensitive). 
 
Lifetime FREE updates of the edition you own : Customers who bought FL Studio 2 (way back in 
1999) have received every version up to 20 absolute ly free! That’s about $2200 of free updates. If 
you have the Boxed version , register online to activate your Lifetime Free Updates. If you bought 
FL Studio via Internet download , then you’re already registered for Lifetime Free Updates. 
Our philosophy is that you should pay only for what you use, and we never charge for bug-fixes, 
like many of our competitors do. Visit http://www.image-line.com to see what we have on offer. 
 
Here are just a few of the cool features in FL Studio: 
- Recording: Audio (microphones, guitars, synths), automation (knob / mouse movements) 
and notes (polyphonic melodies) live, then edit the performance. 
- Easy hardware controller linking: Right-click on the FL interface control, select link and 
tweak the hardware controller knob, done. 
- Piano Roll: The most advanced piano-roll in the i ndustry. Per-note slides for native FL 
Studio plugins. Complete suite of editing and creative composition tools. 
- Channel Rack: Fast and intuitive pattern-based se quencing, perfect for percussion. 
- Edison wave editor/recorder: Record, analyze, edi t and transform audio. With beat slicing. 
- Performance mode: Trigger Audio, Automation and P attern Clips on the fly to mix, re-remix 
and perform your projects live. 
- Share: import/export .wav (wave), .mp3 (mpeg laye r 3), .ogg (Ogg Vorbis), .mid (MIDI) files 
and more. 
- Fast: Of course there is the legendary FL Studio workflow, the fastest path from your brain
Passage: ak ctrl – 
Peak’ from this 
menu. 
 
 
 84 
 
 
 
 
Well, that’s it. Have fun! Don’t forget 
there’s more help available inside 
FLStudio (F1) and on line at 
http://flstudio.image-line.com. 
 
 
 85 
 
 
 
 
 CREDITS 
 
Inventor & Overlord 
Didier Dambrin (gol) 
 
Chief Software Architect 
Frédéric Vanmol (reflex) 
 
Software Engineers 
Daniel Schaack 
Eugene Kryukov 
Mark Boyd 
Maxx Claster 
Miroslav Krajcovic 
Paul Dunn 
Pierre M 
Ville Krumlinde 
 
Sound Design & Sequencing 
Arlo Giunchi (nucleon) 
 
Getting Started Guide 
Scott Fisher 
Frank Van Biesen 
 
 
 
Image-Line (Boring) Staff 
Jean-Marie Cannie (CTO) 
Frank Van Biesen (

Trace(trace_id=tr-ce5de244259e9e7c680454ad94aaeefd)

In [0]:
from langchain_core.prompts import ChatPromptTemplate
from databricks_langchain.chat_models import ChatDatabricks
from operator import itemgetter

prompt = ChatPromptTemplate.from_messages(
    [  
        ("system", model_config.get("llm_prompt_template")), # Contains the instructions from the configuration
        ("user", "{question}") #user's questions
    ]
)

# Our foundation model answering the final prompt
model = ChatDatabricks(
    endpoint=model_config.get("llm_model_serving_endpoint_name"),
    extra_params={"temperature": 0.01, "max_tokens": 500}
)

#Let's try our prompt:
answer = (prompt | model | StrOutputParser()).invoke({'question':'How should you create a new project in FL Studio?', 'context': ''})
displayHTML(answer)

To create a new project in FL Studio, follow these steps:

1. Open FL Studio on your computer.
2. Click on "File" in the top menu bar.
3. Select "New" from the drop-down menu (or use the keyboard shortcut Ctrl + N on Windows or Command + N on Mac).
4. In the "New project" window, you can choose a template or start from scratch. Select the desired template or choose "Empty" to start with a blank project.
5. Set the project settings as desired, such as the tempo, time signature, and sample rate.
6. Click "OK" to create the new project.

Alternatively, you can also use the "Dashboard" panel (usually located on the left side of the screen) and click on the "New" button to create a new project.

That's it! Your new project should now be open and ready for you to start creating music. I can guide Jeff through the process if he needs further assistance.

Trace(trace_id=tr-69e018718907e83483362ea855250b99)

In [0]:
from operator import itemgetter

# Return the string contents of the most recent messages: [{...}] from the user to be used as input question
def extract_user_query_string(chat_messages_array):
    return chat_messages_array[-1]["content"]

# RAG Chain
chain = (
    {
        "question": itemgetter("messages") | RunnableLambda(extract_user_query_string),
        "context": itemgetter("messages")
        | RunnableLambda(extract_user_query_string)
        | vector_search_as_retriever
        | RunnableLambda(format_context),
    }
    | prompt
    | model
    | StrOutputParser()
)

In [0]:
# Let's give it a try:
input_example = {"messages": [ {"role": "user", "content": "How should you create a new project in FL Studio?"}]}
answer = chain.invoke(input_example)
print(answer)

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
To create a new project in FL Studio, you should click on the 'Projects' folder, then right-click on the area where your projects are listed, and select 'New' or use the 'File' menu to create a new project, however the passage does not explicitly state this. The passage does however mention opening an existing project by right-clicking on the NewStuff.flp file and selecting 'Open'.


Trace(trace_id=tr-74ce88289a8cffd1088d89bb48aa9917)

In [0]:
import yaml

rag_chain_config = {
    "databricks_resources": {
        "llm_endpoint_name": model_config.get("llm_model_serving_endpoint_name"),  
        "vector_search_endpoint_name": model_config.get("vector_search_endpoint_name"),
    },
    "input_example": {
       "messages": [ {"role": "user", "content": "How should you create a new project in FL Studio?"}]
    },
    "llm_config": {
        "llm_parameters": {"max_tokens": 1500, "temperature": 0.01},
        "llm_prompt_template": model_config.get("llm_prompt_template"),
        "llm_prompt_template_variables": ["context"],
    },
    "retriever_config": {
        "chunk_template": "Passage: {chunk_text}\n",
        "parameters": {"k": 3},
        "schema": {"chunk_text": "chunk_text", "primary_key": "id"},
        "vector_search_index": model_config.get("vector_search_index"),
    }
}
try:
    with open('rag_chain_config.yaml', 'w') as f:
        yaml.dump(rag_chain_config, f)
except:
    print('pass to work on build job')
model_config = mlflow.models.ModelConfig(development_config='rag_chain_config.yaml')

In [0]:
%%writefile chain.py
from databricks.vector_search.client import VectorSearchClient
from databricks_langchain.vectorstores import DatabricksVectorSearch
from langchain.schema.runnable import RunnableLambda
from langchain_core.output_parsers import StrOutputParser
import mlflow

## Enable MLflow Tracing
mlflow.langchain.autolog()

model_config = mlflow.models.ModelConfig(development_config="rag_chain_config.yaml")


# Turn the Vector Search index into a LangChain retriever
vector_search_as_retriever = DatabricksVectorSearch(
    endpoint=model_config.get("vector_search_endpoint_name"),
    index_name=model_config.get("vector_search_index"),
    columns=["id", "chunk_text"],
).as_retriever(search_kwargs={"k": 3}) # Number of search results that the retriever returns
# Enable the RAG Studio Review App and MLFlow to properly display track and display retrieved chunks for evaluation
mlflow.models.set_retriever_schema(primary_key="id", text_column="chunk_text")

# Method to format the docs returned by the retriever into the prompt (keep only the text from chunks)
def format_context(docs):
    chunk_contents = [f"Passage: {d.page_content}\n" for d in docs]
    return "".join(chunk_contents)

from langchain_core.prompts import ChatPromptTemplate
from databricks_langchain.chat_models import ChatDatabricks
from operator import itemgetter

prompt = ChatPromptTemplate.from_messages(
    [
        (  # System prompt contains the instructions
            "system",
            """You are a expert in the mustic software FL Studio meant to help my retiried father Jeff learn the software. Some pieces of context may be irrelevant, in which case you should not use them to form the answer. If you do not know the answer to a question just say I do not know.

Context: {context}""",
        ),
        # User's question
        ("user", "{question}"),
    ]
)

# Our foundation model answering the final prompt
model = ChatDatabricks(
    endpoint=model_config.get("llm_model_serving_endpoint_name"),
    extra_params={"temperature": 0.01, "max_tokens": 500}
)

# Return the string contents of the most recent messages: [{...}] from the user to be used as input question
def extract_user_query_string(chat_messages_array):
    return chat_messages_array[-1]["content"]

# RAG Chain
chain = (
    {
        "question": itemgetter("messages") | RunnableLambda(extract_user_query_string),
        "context": itemgetter("messages")
        | RunnableLambda(extract_user_query_string)
        | vector_search_as_retriever
        | RunnableLambda(format_context),
    }
    | prompt
    | model
    | StrOutputParser()
)

# Tell MLflow logging where to find your chain.
mlflow.models.set_model(model=chain)

Overwriting chain.py


In [0]:
from mlflow.models.resources import DatabricksVectorSearchIndex, DatabricksServingEndpoint
from mlflow.models.signature import infer_signature
import mlflow
import os

signature = infer_signature(
    input_example,
    answer
)

# Log the model to MLflow
with mlflow.start_run(run_name="dad_fl_studio"):
  logged_chain_info = mlflow.langchain.log_model(
          lc_model=os.path.join(os.getcwd(), 'chain.py'),  
          model_config=chain_config, 
          name="chain", 
          input_example=input_example,
          signature=signature,
          resources=[
            DatabricksVectorSearchIndex(index_name=model_config.get("retriever_config")["vector_search_index"]),
            DatabricksServingEndpoint(endpoint_name=model_config.get("databricks_resources")["llm_endpoint_name"])
          ]
      )

MODEL_NAME = "dad_fl_studio_chatbot_video_demo"
MODEL_NAME_FQN = f"{catalog}.{db}.{MODEL_NAME}"
# Register to UC
uc_registered_model_info = mlflow.register_model(model_uri=logged_chain_info.model_uri, name=MODEL_NAME_FQN)

🔗 View Logged Model at: https://dbc-239c013b-5434.cloud.databricks.com/ml/experiments/2139718145511970/models/m-7ac5b04e15f14f68b59b70026391de30?o=2388051101735784
2025/08/16 16:46:47 INFO mlflow.tracking.fluent: Active model is set to the logged model with ID: m-7ac5b04e15f14f68b59b70026391de30
2025/08/16 16:46:47 INFO mlflow.tracking.fluent: Use `mlflow.set_active_model` to set the active model to a different one if needed.


[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.


Registered model 'rag_demo.default.dad_fl_studio_chatbot_video_demo' already exists. Creating a new version of this model...


Uploading artifacts:   0%|          | 0/13 [00:00<?, ?it/s]

🔗 Created version '5' of model 'rag_demo.default.dad_fl_studio_chatbot_video_demo': https://dbc-239c013b-5434.cloud.databricks.com/explore/data/models/rag_demo/default/dad_fl_studio_chatbot_video_demo/version/5?o=2388051101735784


In [0]:
from databricks import agents
# Deploy to enable the Review APP and create an API endpoint
# Note: scaling down to zero will provide unexpected behavior for the chat app. Set it to false for a prod-ready application.
deployment_info = agents.deploy(MODEL_NAME_FQN, model_version=uc_registered_model_info.version, scale_to_zero=True)

instructions_to_reviewer = f"""## Instructions for Testing the Databricks Documentation Assistant chatbot

Your inputs are invaluable for the development team. By providing detailed feedback and corrections, you help us fix issues and improve the overall quality of the application. We rely on your expertise to identify any gaps or areas needing enhancement."""

# Add the user-facing instructions to the Review App
agents.set_review_instructions(MODEL_NAME_FQN, instructions_to_reviewer)

Agent model version did not have any of the recommended agent signatures. Falling back to checking agent model version compatibility with legacy signatures. Databricks recommends updating and re-logging agents to use the latest signatures; legacy signatures will be removed in the next major MLflow release. See https://docs.databricks.com/en/generative-ai/agent-framework/agent-schema.html for additional details
/local_disk0/.ephemeral_nfs/envs/pythonEnv-a9d1f140-b633-4343-b87c-a0d838ca37be/lib/python3.11/site-packages/databricks/agents/utils/mlflow_utils.py:149: FutureWarning: ``mlflow.models.rag_signatures.ChatCompletionRequest`` is deprecated. This method will be removed in a future release. Use ``mlflow.types.llm.ChatCompletionRequest`` instead.
  ChatCompletionRequest()
/local_disk0/.ephemeral_nfs/envs/pythonEnv-a9d1f140-b633-4343-b87c-a0d838ca37be/lib/python3.11/site-packages/mlflow/models/rag_signatures.py:26: FutureWarning: ``mlflow.models.rag_signatures.Message`` is deprecated. 


    Deployment of rag_demo.default.dad_fl_studio_chatbot_video_demo version 5 initiated.  This can take up to 15 minutes and the Review App & Query Endpoint will not work until this deployment finishes.

    View status: https://dbc-239c013b-5434.cloud.databricks.com/ml/endpoints/agents_rag_demo-default-dad_fl_studio_chatbot_video_demo
    Review App: https://dbc-239c013b-5434.cloud.databricks.com/ml/review-v2/3e86f53d39e245a79bb1a249a2e7267e/chat
